# 07 — Custom DDQN with MarioNet (PyTorch Tutorial)

Custom Double DQN implementation based on the [PyTorch Mario RL tutorial](https://docs.pytorch.org/tutorials/intermediate/mario_rl_tutorial.html).

**Key differences from SB3 DQN (notebook 01):**
- Custom CNN architecture (MarioNet): 3 conv layers + 2 FC layers
- Double DQN: online + target network
- Manual training loop with epsilon-greedy exploration
- TensorDict replay buffer
- gamma=0.9 (short-term focus)
- Epsilon decay: 1.0 → 0.1 over training

In [ ]:
# --- Google Colab Setup ---
import os, sys

if os.getenv('COLAB_RELEASE_TAG'):
    !pip install -q Cython setuptools wheel
    !git clone -b hotfix/version1 https://github.com/lmartim4/CSC-52081-ContinousMountainCar.git /content/repo
    %cd /content/repo
    !pip install -r requirements.txt --no-build-isolation
    !pip install tensordict torchrl
    sys.path.insert(0, '/content/repo')
    import site; SITE = site.getsitepackages()[0]
    !patch --forward -p0 {SITE}/nes_py/_rom.py                   < patches/nes_py_numpy2.patch || true
    !patch --forward -p0 {SITE}/gym/utils/passive_env_checker.py < patches/gym_bool8_numpy2.patch || true
    !patch --forward -p0 {SITE}/gym_super_mario_bros/smb_env.py  < patches/smb_env_numpy2.patch || true
    !sed -i 's/observation, reward, terminated, truncated, info = self.env.step(action)/_result = self.env.step(action); observation, reward, terminated, info = _result[:4]; truncated = _result[4] if len(_result) > 4 else False/' {SITE}/gym/wrappers/time_limit.py
    !git pull
else:
    if os.path.basename(os.getcwd()) == 'notebooks':
        %cd ..

In [ ]:
import torch
import numpy as np
import datetime
from pathlib import Path

from src.wrappers import make_pixel_env
from src.agents.mario_net import Mario, MetricLogger

print(f'Using CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# Create environment (v3 has built-in time limit)
env = make_pixel_env(env_id='SuperMarioBros-1-1-v3', normalize=False)
print(f'Observation space: {env.observation_space.shape}')
print(f'Action space: {env.action_space.n}')

In [ ]:
# Hyperparameters
TOTAL_EPISODES = 1_500
LOG_EVERY = 20

save_dir = Path('../models/custom_ddqn')
save_dir.mkdir(parents=True, exist_ok=True)

# Env gives (84, 84, 4) channels-last -> MarioNet needs (4, 84, 84) channels-first
obs_shape = env.observation_space.shape
state_dim = (obs_shape[2], obs_shape[0], obs_shape[1])  # (4, 84, 84)

mario = Mario(
    state_dim=state_dim,
    action_dim=env.action_space.n,
    save_dir=save_dir,
    lr=0.00025,
    gamma=0.9,
    exploration_rate=1.0,
    exploration_rate_decay=0.99999975,
    exploration_rate_min=0.1,
    batch_size=32,
    buffer_size=100_000,
    burnin=1e4,
    learn_every=3,
    sync_every=1e4,
    save_every=1e5,
)

logger = MetricLogger(save_dir)
print(f'Save directory: {save_dir}')
print(f'State dim: {state_dim}')
print(f'Action dim: {env.action_space.n}')
print(f'Device: {mario.device}')

In [ ]:
%load_ext tensorboard
%tensorboard --logdir ../models/custom_ddqn

In [ ]:
# Training loop
import time

MAX_STEPS = 4000  # max steps per episode (500 * skip=4 = 2000 frames)
t_start = time.time()

for e in range(TOTAL_EPISODES):
    state = env.reset()
    steps = 0

    while True:
        action = mario.act(state)
        next_state, reward, done, truncated, info = env.step(action)

        mario.cache(state, next_state, action, reward, done)
        q, loss = mario.learn()
        logger.log_step(reward, loss, q)

        state = next_state
        steps += 1

        if done or info.get('flag_get', False) or steps >= MAX_STEPS:
            break

    logger.log_episode()

    if (e % LOG_EVERY == 0) or (e == TOTAL_EPISODES - 1):
        elapsed = time.time() - t_start
        eps_per_sec = (e + 1) / elapsed if elapsed > 0 else 0
        eta = (TOTAL_EPISODES - e - 1) / eps_per_sec if eps_per_sec > 0 else 0
        logger.record(episode=e, epsilon=mario.exploration_rate, step=mario.curr_step)
        print(f'  -> Elapsed: {elapsed/60:.1f}min | ETA: {eta/60:.1f}min | {eps_per_sec:.1f} ep/s')

print(f'\nTraining complete in {(time.time()-t_start)/60:.1f} minutes')
print(f'Total steps: {mario.curr_step}')

In [ ]:
# Save final model + training data
import pickle

mario.save()

# Save training metrics for later analysis
metrics = {
    'ep_rewards': logger.ep_rewards,
    'ep_lengths': logger.ep_lengths,
    'ep_avg_losses': logger.ep_avg_losses,
    'ep_avg_qs': logger.ep_avg_qs,
    'total_steps': mario.curr_step,
    'exploration_rate': mario.exploration_rate,
}
with open(save_dir / 'training_metrics.pkl', 'wb') as f:
    pickle.dump(metrics, f)

print(f'Final exploration rate: {mario.exploration_rate:.4f}')
print(f'Models saved in: {save_dir}')

In [ ]:
# Plot training results
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# Rewards
ax = axes[0, 0]
ax.plot(logger.ep_rewards, alpha=0.3, color='blue')
if len(logger.ep_rewards) > 100:
    window = 100
    smoothed = np.convolve(logger.ep_rewards, np.ones(window)/window, mode='valid')
    ax.plot(range(window-1, len(logger.ep_rewards)), smoothed, color='blue', linewidth=2)
ax.set_title('Episode Rewards')
ax.set_xlabel('Episode')

# Lengths
ax = axes[0, 1]
ax.plot(logger.ep_lengths, alpha=0.3, color='orange')
if len(logger.ep_lengths) > 100:
    smoothed = np.convolve(logger.ep_lengths, np.ones(window)/window, mode='valid')
    ax.plot(range(window-1, len(logger.ep_lengths)), smoothed, color='orange', linewidth=2)
ax.set_title('Episode Lengths')
ax.set_xlabel('Episode')

# Loss
ax = axes[1, 0]
ax.plot(logger.ep_avg_losses, alpha=0.3, color='red')
if len(logger.ep_avg_losses) > 100:
    smoothed = np.convolve(logger.ep_avg_losses, np.ones(window)/window, mode='valid')
    ax.plot(range(window-1, len(logger.ep_avg_losses)), smoothed, color='red', linewidth=2)
ax.set_title('Average Loss')
ax.set_xlabel('Episode')

# Q values
ax = axes[1, 1]
ax.plot(logger.ep_avg_qs, alpha=0.3, color='green')
if len(logger.ep_avg_qs) > 100:
    smoothed = np.convolve(logger.ep_avg_qs, np.ones(window)/window, mode='valid')
    ax.plot(range(window-1, len(logger.ep_avg_qs)), smoothed, color='green', linewidth=2)
ax.set_title('Average Q Value')
ax.set_xlabel('Episode')

plt.tight_layout()
plt.savefig(str(save_dir / 'training_curves.png'), dpi=150)
plt.show()

In [ ]:
# Quick evaluation
mario.exploration_rate = 0  # Greedy evaluation
rewards = []
flags = []

for ep in range(10):
    state = env.reset()
    total_reward = 0
    done = False
    flag = False

    while not done:
        action = mario.act(state)
        state, reward, done, truncated, info = env.step(action)
        total_reward += reward
        if info.get('flag_get', False):
            flag = True

    rewards.append(total_reward)
    flags.append(flag)

print(f'Mean reward: {np.mean(rewards):.1f} ± {np.std(rewards):.1f}')
print(f'Flag rate: {np.mean(flags):.2%}')